In [1]:
import pandas as pd
import numpy as np

# Load merged dataset
df = pd.read_csv("Documents/military_cleaned.csv")

def clean_numeric(series):
    return (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.extract(r"([-+]?\d*\.?\d+)")[0]
        .astype(float)
    )


numeric_columns = [
    "active_personnel",
    "reserve_personnel",
    "paramilitary",
    "fighter_aircraft",
    "attack_aircraft",
    "total_military_aircraft",
    "total_military_helicopters",
    "tanks",
    "armored_fighting_vehicles",
    "self_propelled_artillery",
    "towed_artillery",
    "rocket_projectors",
    "total_naval_fleet",
    "defense_budget_usd",
    "total_population",
    "gdp_usd",
    "roadway_coverage_km",
    "railway_coverage_km",
    "major_ports_and_terminals",
    "total_serviceable_airports",
    "power_index",
    "power_index_rank"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = clean_numeric(df[col])

print("Numeric columns cleaned successfully.")

Numeric columns cleaned successfully.


In [15]:
# -----------------------------------
# Combined Personnel
# -----------------------------------
df["combined_personnel"] = (
    df["active_personnel"]
    + df["reserve_personnel"]
    + df["paramilitary"]
)

# -----------------------------------
# Total Military Assets
# -----------------------------------
asset_cols = [
    "total_military_aircraft",
    "total_military_helicopters",
    "tanks",
    "armored_fighting_vehicles",
    "self_propelled_artillery",
    "towed_artillery",
    "rocket_projectors",
    "total_naval_fleet"
]

df["total_military_assets"] = df[asset_cols].sum(axis=1)

# -----------------------------------
# Force Readiness Index
# -----------------------------------
df["force_readiness_index"] = (
    df["active_personnel"] /
    df["combined_personnel"]
)

# -----------------------------------
# Air Superiority Index
# -----------------------------------
df["air_superiority_index"] = (
    df["fighter_aircraft"] +
    df["attack_aircraft"]
) / df["total_military_aircraft"]

# -----------------------------------
# Logistics Index
# -----------------------------------
logistics_cols = [
    "roadway_coverage_km",
    "railway_coverage_km",
    "major_ports_and_terminals",
    "total_serviceable_airports"
]

# Normalize logistics columns
for col in logistics_cols:
    df[col] = (
        (df[col] - df[col].min()) /
        (df[col].max() - df[col].min())
    )

df["logistics_index"] = df[logistics_cols].mean(axis=1)

# -----------------------------------
# Superiority Index
# -----------------------------------
asset_norm = (
    (df["total_military_assets"] - df["total_military_assets"].min()) /
    (df["total_military_assets"].max() - df["total_military_assets"].min())
)

personnel_norm = (
    (df["combined_personnel"] - df["combined_personnel"].min()) /
    (df["combined_personnel"].max() - df["combined_personnel"].min())
)

df["superiority_index"] = (
    0.6 * asset_norm +
    0.4 * personnel_norm
)

# -----------------------------------
# Power Index Rank Gap
# -----------------------------------
df = df.sort_values("power_index_rank")

df["power_index_rank_gap"] = (
    df["power_index_rank"]
    .diff()
    .fillna(0)
)

# -----------------------------------
# Coalition Score
# -----------------------------------
df["coalition_score"] = np.where(
    df["alliance"] == "NATO",
    100,
    50
)

# -----------------------------------
# Coalition Budget
# -----------------------------------
df["coalition_budget"] = np.where(
    df["alliance"] == "NATO",
    df["defense_budget_usd"],
    0
)

# -----------------------------------
# Power Advantage
# -----------------------------------
df["power_advantage"] = (
    df["superiority_index"] /
    df["superiority_index"].mean()
)

# -----------------------------------
# Budget Advantage
# -----------------------------------
df["budget_advantage"] = (
    df["defense_budget_usd"] /
    df["defense_budget_usd"].mean()
)

# -----------------------------------
# Assets Per Capita
# -----------------------------------
df["assets_per_capita"] = (
    df["total_military_assets"] /
    df["total_population"]
)

# -----------------------------------
# Budget to GDP Ratio (%)
# -----------------------------------
df["budget_to_gdp_ratio"] = (
    df["defense_budget_usd"] /
    df["gdp_usd"]
) * 100

In [4]:
population = pd.read_csv("total_population.csv")

In [5]:
print(population.columns)

Index(['Country', 'total_population'], dtype='object')


In [6]:
population.rename(columns={
    "Country": "country"
}, inplace=True)

In [7]:
population["country"] = population["country"].str.strip()
df["country"] = df["country"].str.strip()

In [8]:
df = df.merge(
    population[["country", "total_population"]],
    on="country",
    how="left"
)

In [9]:
print(df[["country", "total_population"]].head())
print(df["total_population"].isnull().sum())

         country total_population
0  United States      341,963,408
1         Russia      140,820,810
2          China    1,415,043,270
3          India    1,409,128,296
4    South Korea       52,081,799
3


In [10]:
print(df["total_population"].dtype)

object


In [11]:
df["total_population"] = (
    df["total_population"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace(" ", "", regex=False)
)

df["total_population"] = pd.to_numeric(
    df["total_population"],
    errors="coerce"
)

In [12]:
print(df["total_population"].dtype)
print(df["total_population"].head())

float64
0    3.419634e+08
1    1.408208e+08
2    1.415043e+09
3    1.409128e+09
4    5.208180e+07
Name: total_population, dtype: float64


In [13]:
df["assets_per_capita"] = (
    df["total_military_assets"] /
    df["total_population"]
)

In [14]:
df.to_csv("military_final.csv", index=False)
df.to_excel("military_final.xlsx", index=False)

print("Final dataset saved successfully!")

Final dataset saved successfully!
